# Walmart Sales Real-Time Pipeline: Model Training
This notebook covers the exploratory data analysis (EDA), feature engineering, and training of a Random Forest model to predict `Weekly_Sales`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os

### 1. Load Data and EDA

In [ ]:
# Load data
df = pd.read_csv('../data/Walmart.csv')
display(df.head())

# Check nulls and datatypes
print("\n--- Null Values ---")
print(df.isnull().sum())
print("\n--- Data Types ---")
print(df.dtypes)

In [ ]:
# Weekly_Sales distribution plot
plt.figure(figsize=(10, 6))
sns.histplot(df['Weekly_Sales'], bins=50, kde=True)
plt.title('Weekly Sales Distribution')
plt.show()

In [ ]:
# Correlation heatmap (numeric columns only)
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

### 2. Feature Engineering

In [ ]:
# Extract Week, Month, Year from Date
# Assuming format is DD-MM-YYYY, you can adjust if the CSV uses different format (e.g. YYYY-MM-DD)
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)

# Encode IsHoliday
df['IsHoliday'] = df['IsHoliday'].astype(int)
display(df.head())

### 3. Model Training

In [ ]:
features = ['Store', 'Dept', 'Week', 'Month', 'Year', 'IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
X = df[features]
y = df['Weekly_Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

### 4. Evaluation

In [ ]:
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")

### 5. Save Model

In [ ]:
os.makedirs('../models', exist_ok=True)
model_path = '../models/sales_model.pkl'
joblib.dump(model, model_path)
print(f"Model successfully saved to {model_path}")